# RAG-ассистент Фильмоскоп

Минимальный проектный notebook. Текущая реализация закрывает шаги 3–4 плана: два способа чанкинга, эмбеддинги, FAISS на CPU по умолчанию, baseline retrieval, retrieval-метрики, выбор лучшей стратегии и диагностику candidate pool. GPU FAISS используется только опционально, если он доступен в окружении.


## Константы и импорты


In [1]:
import json
import re
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_PATH = ROOT / "filmoskop_kb.json"
EMBED_MODEL = "intfloat/multilingual-e5-base"
RECALL_KS = [1, 3, 5, 10, 20]
TOP_K = 20

## Проверка зависимостей для шага 3


In [2]:
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Не установлена зависимость: langchain-text-splitters. "
        "Установите её в том же окружении/Jupyter kernel: "
        "python -m pip install langchain-text-splitters"
    ) from e

from sentence_transformers import SentenceTransformer
import faiss
import torch

print("Версия FAISS:", getattr(faiss, "__version__", "unknown"))

FAISS_GPU_AVAILABLE = (
    hasattr(faiss, "StandardGpuResources")
    and hasattr(faiss, "get_num_gpus")
    and faiss.get_num_gpus() > 0
)

if FAISS_GPU_AVAILABLE:
    print("Доступно GPU для FAISS:", faiss.get_num_gpus())
else:
    print("GPU FAISS недоступен; используется CPU FAISS.")

TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Устройство PyTorch для эмбеддингов:", TORCH_DEVICE)


Версия FAISS: 1.16.0
GPU FAISS недоступен; используется CPU FAISS.
Устройство PyTorch для эмбеддингов: cuda


## Диагностика датасета и базовая очистка для шага 3

Эта ячейка загружает базу знаний для локального тестового прогона. Простая очистка и сводка по данным нужны перед чанкингом и построением индекса поиска.


In [3]:
with DATA_PATH.open(encoding="utf-8") as f:
    data = json.load(f)

docs = data["docs"]
queries = data["queries"]

print("Документы:", len(docs), "| Вопросы:", len(queries))
print("Форматы:", dict(Counter(d["format"] for d in docs)))
print("Категории:", dict(Counter(d["category"] for d in docs)))
print("Типы вопросов:", dict(Counter(q["kind"] for q in queries)))


Документы: 181 | Вопросы: 66
Форматы: {'text': 132, 'markdown': 28, 'html': 21}
Категории: {'Подписка и оплата': 38, 'Функции и контент': 34, 'Аккаунт и профили': 35, 'Устройства и качество': 42, 'Ошибки воспроизведения': 32}
Типы вопросов: {'exact': 15, 'semantic': 25, 'multi': 16, 'gap': 10}


In [4]:
def clean_text(raw: str) -> str:
    """Очищает исходный документ и нормализует пробелы.

    Args:
        raw: Исходный текст документа из базы знаний.

    Returns:
        Очищенный текст без простых HTML/Markdown-разметок и лишних пробелов.
    """
    text = raw or ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", " ", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"[`*_#>\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

clean_docs = []
for d in docs:
    clean_docs.append({
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "text": clean_text(d["raw"]),
    })

pd.DataFrame({
    "metric": ["docs", "queries", "median_clean_len"],
    "value": [len(clean_docs), len(queries), int(np.median([len(d["text"]) for d in clean_docs]))],
})


,metric,value
0,docs,181
1,queries,66
2,median_clean_len,662


## Шаг 3. Два варианта чанкинга


In [5]:
CHUNK_STRATEGIES = {
    "small_500_100": {"chunk_size": 500, "chunk_overlap": 100},
    "medium_900_150": {"chunk_size": 900, "chunk_overlap": 150},
}

def make_chunks(clean_docs, chunk_size: int, chunk_overlap: int):
    """Делит тексты на чанки с метаданными документов.

    Args:
        clean_docs: Список очищенных документов с ключами `doc_id`, `title`,
            `category` и `text`.
        chunk_size: Максимальный размер чанка в символах.
        chunk_overlap: Количество символов перекрытия между чанками.

    Returns:
        Список словарей с текстом чанка и метаданными `doc_id`, `title`,
        `category`, `chunk_id`.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in clean_docs:
        parts = splitter.split_text(f"{doc['title']}\n{doc['text']}")
        for i, part in enumerate(parts):
            chunks.append({
                "page_content": part,
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "category": doc["category"],
                "chunk_id": i,
            })
    return chunks

chunk_sets = {}
for name, params in CHUNK_STRATEGIES.items():
    chunk_sets[name] = make_chunks(clean_docs, **params)
    lengths = [len(c["page_content"]) for c in chunk_sets[name]]
    print(name, "чанков:", len(lengths), "медианная длина:", int(np.median(lengths)))


small_500_100 чанков: 502 медианная длина: 374
medium_900_150 чанков: 323 медианная длина: 462


## Шаг 3. Эмбеддинги и FAISS-индексы


In [6]:
embedder = SentenceTransformer(EMBED_MODEL, device=TORCH_DEVICE)

def encode_passages(texts):
    """Создаёт векторы документов для модели E5.

    Args:
        texts: Список текстов чанков.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["passage: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=True,
    )
    return np.asarray(vecs, dtype="float32")

def encode_queries(texts):
    """Создаёт нормализованные векторы запросов для модели E5.

    Args:
        texts: Список текстов запросов.

    Returns:
        Массив эмбеддингов `float32`, нормализованных для поиска по
        скалярному произведению в FAISS.
    """
    vecs = embedder.encode(
        ["query: " + t for t in texts],
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    )
    return np.asarray(vecs, dtype="float32")

def build_faiss_index(vectors):
    """Создаёт индекс FAISS `IndexFlatIP` для нормализованных эмбеддингов.

    Args:
        vectors: Матрица dense-эмбеддингов `float32` формы
            `(количество_чанков, размерность_эмбеддинга)`.

    Returns:
        Индекс FAISS с добавленными векторами. По умолчанию используется CPU
        FAISS; GPU FAISS используется только если он доступен в окружении.
    """
    cpu_index = faiss.IndexFlatIP(vectors.shape[1])
    if FAISS_GPU_AVAILABLE:
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
        index.add(vectors)
        return index

    cpu_index.add(vectors)
    return cpu_index

indexes = {}
embeddings = {}
for name, chunks in chunk_sets.items():
    embeddings[name] = encode_passages([c["page_content"] for c in chunks])
    indexes[name] = build_faiss_index(embeddings[name])
    print(name, embeddings[name].shape, "размер индекса", indexes[name].ntotal)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

small_500_100 (502, 768) размер индекса 502


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

medium_900_150 (323, 768) размер индекса 323


## Шаг 3. Retrieval-метрики


In [7]:
eval_queries = [q for q in queries if q["kind"] != "gap"]
query_vecs = encode_queries([q["query"] for q in eval_queries])

def retrieved_doc_ids(strategy: str, q_vec, k: int):
    """Возвращает уникальные `doc_id` из top-k найденных чанков.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.
        q_vec: Вектор запроса.
        k: Количество чанков, запрашиваемое у FAISS.

    Returns:
        Список уникальных идентификаторов документов в порядке релевантности.
    """
    _, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    doc_ids = []
    seen = set()
    for i in idx[0]:
        doc_id = chunk_sets[strategy][int(i)]["doc_id"]
        if doc_id not in seen:
            doc_ids.append(doc_id)
            seen.add(doc_id)
    return doc_ids

def precision_at_k(found, relevant):
    """Считает precision для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля найденных документов, которые входят в список релевантных.
    """
    if not found:
        return 0.0
    rel = set(relevant)
    return sum(doc_id in rel for doc_id in found) / len(found)

def recall_at_k(found, relevant):
    """Считает recall для найденных `doc_id`.

    Args:
        found: Список найденных идентификаторов документов.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Доля релевантных документов, найденных поиском.
    """
    rel = set(relevant)
    if not rel:
        return 0.0
    return len(set(found) & rel) / len(rel)

def mrr(found, relevant):
    """Считает reciprocal rank первого релевантного документа.

    Args:
        found: Список найденных идентификаторов документов в порядке релевантности.
        relevant: Список эталонных идентификаторов документов.

    Returns:
        Значение reciprocal rank или `0.0`, если релевантных документов нет в выдаче.
    """
    rel = set(relevant)
    for rank, doc_id in enumerate(found, start=1):
        if doc_id in rel:
            return 1 / rank
    return 0.0

def evaluate_strategy(strategy: str):
    """Считает retrieval-метрики для одной стратегии чанкинга.

    Args:
        strategy: Название стратегии чанкинга из `chunk_sets` и `indexes`.

    Returns:
        `pandas.DataFrame` по вопросам со списками найденных и релевантных документов.
    """
    rows = []
    for q, q_vec in zip(eval_queries, query_vecs):
        found20 = retrieved_doc_ids(strategy, q_vec, TOP_K)
        row = {
            "strategy": strategy,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found20,
            "precision@5": precision_at_k(found20[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found20[:5], q["relevant_docs"]),
            "MRR": mrr(found20, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found20[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

metrics_by_query = pd.concat([evaluate_strategy(name) for name in chunk_sets], ignore_index=True)
summary = metrics_by_query.groupby("strategy")[["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]].mean().reset_index()
summary


,strategy,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,medium_900_150,0.146429,0.562500,0.417792,0.169643,0.5,0.669643,0.785714
1,small_500_100,0.153571,0.589286,0.465340,0.258929,0.5,0.687500,0.776786


In [8]:
by_kind = metrics_by_query.groupby(["strategy", "kind"])[["precision@5", "recall@5", "MRR"]].mean().reset_index()
by_kind

,strategy,kind,precision@5,recall@5,MRR
0,medium_900_150,exact,0.173333,0.833333,0.497222
1,medium_900_150,multi,0.200000,0.500000,0.472070
2,medium_900_150,semantic,0.096000,0.440000,0.335397
3,small_500_100,exact,0.173333,0.833333,0.638889
4,small_500_100,multi,0.212500,0.531250,0.520908
5,small_500_100,semantic,0.104000,0.480000,0.325648


## Шаг 3. Анализ примеров чанков и выбор стратегии


In [9]:
for name, chunks in chunk_sets.items():
    print("\n===", name, "===")
    for c in chunks[:3]:
        print(f"doc_id={c['doc_id']} chunk_id={c['chunk_id']} title={c['title']}")
        print(c["page_content"][:450].replace("\n", " "))
        print("---")


=== small_500_100 ===
doc_id=0 chunk_id=0 title=Промокоды
Промокоды Промокод даёт скидку на подписку или бесплатные дни. Ввести его можно в разделе «Подписка» — поле «У меня есть промокод» — или при оформлении подписки. Правила простые: один промокод на один аккаунт, скидки по промокодам не суммируются между собой и с другими акциями, у каждого кода есть срок действия — он указан в условиях акции. Если код не применяется, проверьте срок действия и то, что он ещё не был использован на этом аккаунте.
---
doc_id=1 chunk_id=0 title=Синхронизация между устройствами
Синхронизация между устройствами
---
doc_id=1 chunk_id=1 title=Синхронизация между устройствами
Всё, что вы делаете в «Фильмоскопе», хранится не на устройстве, а в аккаунте. Поэтому телефон, планшет, браузер и телевизор показывают одно и то же: вошли — и сервис выглядит привычно, даже если устройство чужое. Общее для аккаунта: подписка и тариф, способ оплаты, список привязанных устройств. Это не зависит ни от профиля, ни от

In [10]:
best_strategy = summary.sort_values(["recall@5", "MRR", "precision@5"], ascending=False).iloc[0]["strategy"]
print("Выбранная стратегия чанкинга:", best_strategy)
print(summary.to_string(index=False))


Выбранная стратегия чанкинга: small_500_100
      strategy  precision@5  recall@5      MRR  recall@1  recall@3  recall@10  recall@20
medium_900_150     0.146429  0.562500 0.417792  0.169643       0.5   0.669643   0.785714
 small_500_100     0.153571  0.589286 0.465340  0.258929       0.5   0.687500   0.776786


### Вывод по шагу 3

После локального запуска нужно сравнить агрегированные метрики `summary` и `by_kind`, выбрать стратегию по `recall@5`, `MRR`, `precision@5` и зафиксировать причины выбора. Выбранная стратегия используется в шаге 4 для baseline retrieval и диагностики ошибок. Финальный контрольный прогон выполняется на удалённой машине только после успешного локального запуска.


## Шаг 4. Baseline retrieval и диагностика

На выбранной стратегии чанкинга собираем итоговый dense baseline: выдачу top-K чанков, распределение score для подбора порога, doc-level overfetch с агрегацией чанков в уникальные `doc_id`, пример фильтрации по категории и диагностику ошибок по типам запросов.

In [11]:
BASELINE_CHUNK_TOP_K = 20
OVERFETCH_CHUNK_KS = [50, 100]
DOC_TOP_K = 20
FILTER_CATEGORY = "Подписка и оплата"

selected_strategy = best_strategy
selected_chunks = chunk_sets[selected_strategy]
selected_index = indexes[selected_strategy]
doc_by_id = {doc["doc_id"]: doc for doc in clean_docs}

eval_query_rows = list(zip(eval_queries, query_vecs))

print("Стратегия для baseline retrieval:", selected_strategy)
print("Чанков в выбранной стратегии:", len(selected_chunks))
print("Eval-запросов без gap:", len(eval_query_rows))

Стратегия для baseline retrieval: small_500_100
Чанков в выбранной стратегии: 502
Eval-запросов без gap: 56


In [12]:
def dense_chunk_search(strategy: str, q_vec, chunk_k: int, category: str | None = None, score_threshold: float | None = None):
    """Searches chunks with FAISS and returns results with metadata.

    Args:
        strategy: Chunking strategy name from `chunk_sets` and `indexes`.
        q_vec: Query embedding vector.
        chunk_k: Number of chunk candidates requested from FAISS.
        category: Optional category for post-filtering results.
        score_threshold: Optional minimal dense score.

    Returns:
        List of dictionaries with rank, score, doc_id, title, category, chunk_id and page_content.
    """
    chunks = chunk_sets[strategy]
    k = min(chunk_k, len(chunks))
    scores, idx = indexes[strategy].search(q_vec.reshape(1, -1), k)
    hits = []
    rank = 0
    for score, chunk_idx in zip(scores[0], idx[0]):
        if chunk_idx < 0:
            continue
        chunk = chunks[int(chunk_idx)]
        score = float(score)
        if category is not None and chunk["category"] != category:
            continue
        if score_threshold is not None and score < score_threshold:
            continue
        rank += 1
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def aggregate_chunks_to_docs(chunk_hits, doc_k: int | None = None):
    """Aggregates chunk-level results into unique documents by best chunk score.

    Args:
        chunk_hits: Results returned by `dense_chunk_search`.
        doc_k: Optional maximum number of documents in the output.

    Returns:
        Ranked list of unique documents sorted by the best chunk score.
    """
    docs_map = {}
    for hit in chunk_hits:
        doc_id = hit["doc_id"]
        if doc_id not in docs_map:
            docs_map[doc_id] = {
                "doc_id": doc_id,
                "title": hit["title"],
                "category": hit["category"],
                "best_score": hit["score"],
                "best_chunk_rank": hit["rank"],
                "best_chunk_id": hit["chunk_id"],
                "chunk_count": 1,
                "page_content": hit["page_content"],
            }
        else:
            docs_map[doc_id]["chunk_count"] += 1
            if hit["score"] > docs_map[doc_id]["best_score"]:
                docs_map[doc_id].update({
                    "best_score": hit["score"],
                    "best_chunk_rank": hit["rank"],
                    "best_chunk_id": hit["chunk_id"],
                    "page_content": hit["page_content"],
                })
    docs_ranked = sorted(
        docs_map.values(),
        key=lambda row: (-row["best_score"], row["best_chunk_rank"]),
    )
    if doc_k is not None:
        docs_ranked = docs_ranked[:doc_k]
    for rank, row in enumerate(docs_ranked, start=1):
        row["doc_rank"] = rank
    return docs_ranked


def dense_doc_ids_with_overfetch(strategy: str, q_vec, chunk_k: int, doc_k: int):
    """Returns top document ids after overfetching chunk candidates.

    Args:
        strategy: Chunking strategy name from `chunk_sets` and `indexes`.
        q_vec: Query embedding vector.
        chunk_k: Number of chunk candidates requested from FAISS.
        doc_k: Number of final unique documents.

    Returns:
        List of `doc_id` values after chunk aggregation.
    """
    chunk_hits = dense_chunk_search(strategy, q_vec, chunk_k)
    doc_hits = aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)
    return [row["doc_id"] for row in doc_hits]


def evaluate_dense_variant(strategy: str, chunk_k: int, doc_k: int, variant: str):
    """Computes retrieval metrics for a dense variant with selected overfetch.

    Args:
        strategy: Chunking strategy name from `chunk_sets` and `indexes`.
        chunk_k: Number of chunk candidates requested from FAISS.
        doc_k: Number of final documents after aggregation.
        variant: Human-readable variant name.

    Returns:
        `pandas.DataFrame` with per-query metrics.
    """
    rows = []
    for q, q_vec in eval_query_rows:
        found_docs = dense_doc_ids_with_overfetch(strategy, q_vec, chunk_k=chunk_k, doc_k=doc_k)
        row = {
            "variant": variant,
            "strategy": strategy,
            "chunk_k": chunk_k,
            "doc_k": doc_k,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)

In [13]:
score_rows = []
for query_idx, (q, q_vec) in enumerate(eval_query_rows):
    for hit in dense_chunk_search(selected_strategy, q_vec, chunk_k=BASELINE_CHUNK_TOP_K):
        score_rows.append({
            "query_idx": query_idx,
            "kind": q["kind"],
            "chunk_rank": hit["rank"],
            "score": hit["score"],
            "doc_id": hit["doc_id"],
            "is_relevant_doc": hit["doc_id"] in set(q["relevant_docs"]),
        })

score_distribution = pd.DataFrame(score_rows)
score_quantiles = score_distribution["score"].quantile([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).reset_index()
score_quantiles.columns = ["quantile", "score"]
DENSE_SCORE_THRESHOLD = float(score_distribution["score"].quantile(0.10))

print("Диагностический dense-score threshold p10:", round(DENSE_SCORE_THRESHOLD, 4))
score_quantiles

Диагностический dense-score threshold p10: 0.8191


,quantile,score
0,0.01,0.799701
1,0.05,0.812193
2,0.10,0.819056
3,0.25,0.826536
4,0.50,0.836262
5,0.75,0.846925
6,0.90,0.858379
7,0.95,0.865907
8,0.99,0.885363


In [14]:
sample_query, sample_vec = eval_query_rows[0]
sample_chunk_hits = dense_chunk_search(selected_strategy, sample_vec, chunk_k=10)
sample_doc_hits = aggregate_chunks_to_docs(sample_chunk_hits, doc_k=5)

print("Пример запроса:", sample_query["query"])
print("Эталонные doc_id:", sample_query["relevant_docs"])
pd.DataFrame(sample_chunk_hits)[["rank", "score", "doc_id", "title", "category", "chunk_id"]].head(10)

Пример запроса: что означает ошибка P-102
Эталонные doc_id: [85]


,rank,score,doc_id,title,category,chunk_id
0,1,0.885515,70,Ошибка P-102 (архив),Ошибки воспроизведения,0
1,2,0.874992,70,Ошибка P-102 (архив),Ошибки воспроизведения,1
2,3,0.868469,23,Ошибка P-107 (браузер),Ошибки воспроизведения,0
3,4,0.865473,160,Ошибка P-107 в браузере,Ошибки воспроизведения,0
4,5,0.861530,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0
5,6,0.856184,138,Ошибка P-104: устройство не поддерживает защит...,Ошибки воспроизведения,1
6,7,0.853294,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,1
7,8,0.852270,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,0
8,9,0.851573,129,Ошибка P-105: устаревшее приложение,Ошибки воспроизведения,1
9,10,0.849812,137,Ошибка P-106: сбой загрузки субтитров,Ошибки воспроизведения,1


In [15]:
print("Top-5 документов после агрегации chunk hits:")
pd.DataFrame(sample_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Top-5 документов после агрегации chunk hits:


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.885515,70,Ошибка P-102 (архив),Ошибки воспроизведения,0,2
1,2,0.868469,23,Ошибка P-107 (браузер),Ошибки воспроизведения,0,1
2,3,0.865473,160,Ошибка P-107 в браузере,Ошибки воспроизведения,0,1
3,4,0.861530,85,Ошибка P-102: превышен лимит просмотров,Ошибки воспроизведения,0,2
4,5,0.856184,138,Ошибка P-104: устройство не поддерживает защит...,Ошибки воспроизведения,1,1


In [16]:
baseline_variants = []
baseline_variants.append(evaluate_dense_variant(
    selected_strategy,
    chunk_k=BASELINE_CHUNK_TOP_K,
    doc_k=DOC_TOP_K,
    variant="dense_chunk20_doc20",
))
for chunk_k in OVERFETCH_CHUNK_KS:
    baseline_variants.append(evaluate_dense_variant(
        selected_strategy,
        chunk_k=chunk_k,
        doc_k=DOC_TOP_K,
        variant=f"dense_chunk{chunk_k}_doc20",
    ))

baseline_diagnostics = pd.concat(baseline_variants, ignore_index=True)
baseline_summary = baseline_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@20", "recall@5", "MRR"], ascending=False)
baseline_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,0.153571,0.589286,0.467225,0.258929,0.5,0.6875,0.830357
2,dense_chunk50_doc20,0.153571,0.589286,0.467225,0.258929,0.5,0.6875,0.830357
1,dense_chunk20_doc20,0.153571,0.589286,0.465340,0.258929,0.5,0.6875,0.776786


In [17]:
baseline_by_kind = baseline_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
baseline_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_chunk100_doc20,exact,0.173333,0.833333,0.638889,0.466667,0.766667,0.933333,0.933333
1,dense_chunk100_doc20,multi,0.212500,0.531250,0.520908,0.156250,0.437500,0.718750,0.843750
2,dense_chunk100_doc20,semantic,0.104000,0.480000,0.329871,0.200000,0.380000,0.520000,0.760000
3,dense_chunk20_doc20,exact,0.173333,0.833333,0.638889,0.466667,0.766667,0.933333,0.933333
4,dense_chunk20_doc20,multi,0.212500,0.531250,0.520908,0.156250,0.437500,0.718750,0.781250
5,dense_chunk20_doc20,semantic,0.104000,0.480000,0.325648,0.200000,0.380000,0.520000,0.680000
6,dense_chunk50_doc20,exact,0.173333,0.833333,0.638889,0.466667,0.766667,0.933333,0.933333
7,dense_chunk50_doc20,multi,0.212500,0.531250,0.520908,0.156250,0.437500,0.718750,0.843750
8,dense_chunk50_doc20,semantic,0.104000,0.480000,0.329871,0.200000,0.380000,0.520000,0.760000


In [18]:
category_query_idx = None
for i, (q, q_vec) in enumerate(eval_query_rows):
    relevant_categories = {doc_by_id[doc_id]["category"] for doc_id in q["relevant_docs"] if doc_id in doc_by_id}
    if FILTER_CATEGORY in relevant_categories:
        category_query_idx = i
        break

if category_query_idx is None:
    category_query_idx = 0

category_query, category_vec = eval_query_rows[category_query_idx]
category_hits = dense_chunk_search(
    selected_strategy,
    category_vec,
    chunk_k=100,
    category=FILTER_CATEGORY,
    score_threshold=None,
)
category_doc_hits = aggregate_chunks_to_docs(category_hits, doc_k=5)

print("Категория фильтра:", FILTER_CATEGORY)
print("Запрос:", category_query["query"])
print("Эталонные doc_id:", category_query["relevant_docs"])
pd.DataFrame(category_doc_hits)[["doc_rank", "best_score", "doc_id", "title", "category", "best_chunk_id", "chunk_count"]]

Категория фильтра: Подписка и оплата
Запрос: сколько стоит тариф Максимум и что в него входит
Эталонные doc_id: [113]


,doc_rank,best_score,doc_id,title,category,best_chunk_id,chunk_count
0,1,0.864235,113,Тарифы «Фильмоскопа»,Подписка и оплата,3,5
1,2,0.854001,88,Смена тарифа (архив),Подписка и оплата,1,3
2,3,0.846702,24,Какой тариф подойдёт именно вам,Подписка и оплата,1,4
3,4,0.839782,25,Тарифы «Фильмоскопа» (архив),Подписка и оплата,1,2
4,5,0.834123,29,Как меняются цены,Подписка и оплата,2,3


In [19]:
diagnostic_variant = "dense_chunk100_doc20" if "dense_chunk100_doc20" in set(baseline_diagnostics["variant"]) else baseline_summary.iloc[0]["variant"]
problem_rows = []

for q, q_vec in eval_query_rows:
    chunk_hits_100 = dense_chunk_search(selected_strategy, q_vec, chunk_k=100)
    doc_hits_100 = aggregate_chunks_to_docs(chunk_hits_100, doc_k=100)
    doc_positions = {row["doc_id"]: row["doc_rank"] for row in doc_hits_100}
    found20 = [row["doc_id"] for row in doc_hits_100[:DOC_TOP_K]]
    relevant = q["relevant_docs"]
    relevant_positions = {doc_id: doc_positions.get(doc_id) for doc_id in relevant}
    missing_relevant = [doc_id for doc_id, pos in relevant_positions.items() if pos is None]
    below_top5 = [doc_id for doc_id, pos in relevant_positions.items() if pos is not None and pos > 5]
    recall5 = recall_at_k(found20[:5], relevant)
    recall20 = recall_at_k(found20[:20], relevant)

    reasons = []
    if missing_relevant:
        reasons.append("relevant docs отсутствуют в top-100 candidate pool")
    if below_top5:
        reasons.append("relevant docs есть ниже top-5")
    if q["kind"] == "semantic":
        reasons.append("возможна сильная переформулировка запроса")
    if q["kind"] == "exact":
        reasons.append("dense-поиск может плохо ловить точные термины/коды")
    if q["kind"] == "multi":
        reasons.append("multi-запрос может требовать декомпозиции")

    if recall5 < 1.0:
        problem_rows.append({
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": relevant,
            "found_top5": found20[:5],
            "found_top20": found20[:20],
            "recall@5": recall5,
            "recall@20": recall20,
            "relevant_positions_in_top100_docs": relevant_positions,
            "diagnosis": "; ".join(reasons),
        })

problem_queries = pd.DataFrame(problem_rows).sort_values(["recall@5", "recall@20", "kind"]).reset_index(drop=True)
print("Проблемных запросов с recall@5 < 1.0:", len(problem_queries))
problem_queries.head(15)

Проблемных запросов с recall@5 < 1.0: 29


,kind,query,relevant_docs,found_top5,found_top20,recall@5,recall@20,relevant_positions_in_top100_docs,diagnosis
0,exact,как подключить саундбар по HDMI ARC,[22],"[28, 110, 102, 47, 125]","[28, 110, 102, 47, 125, 69, 99, 70, 168, 152, ...",0.0,0.0,{22: 22},relevant docs есть ниже top-5; dense-поиск мож...
1,semantic,"подключил колонки, но объёмного звука нет — ид...",[22],"[28, 112, 50, 71, 102]","[28, 112, 50, 71, 102, 52, 110, 69, 70, 99, 15...",0.0,0.0,{22: 24},relevant docs есть ниже top-5; возможна сильна...
2,semantic,"как посмотреть, за что с меня списали деньги в...",[106],"[5, 114, 52, 162, 64]","[5, 114, 52, 162, 64, 71, 164, 154, 173, 116, ...",0.0,0.0,{106: 31},relevant docs есть ниже top-5; возможна сильна...
3,semantic,"хочу убрать из ленты фильмы, которые уже смотрел",[97],"[81, 20, 121, 62, 110]","[81, 20, 121, 62, 110, 99, 71, 157, 47, 102, 1...",0.0,0.0,{97: 41},relevant docs есть ниже top-5; возможна сильна...
4,semantic,"не помню название, помню только актёра",[158],"[71, 159, 46, 52, 178]","[71, 159, 46, 52, 178, 124, 112, 22, 38, 111, ...",0.0,0.0,{158: None},relevant docs отсутствуют в top-100 candidate ...
5,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[72, 41, 47, 102, 117]","[72, 41, 47, 102, 117, 94, 54, 177, 129, 137, ...",0.0,0.0,{124: 47},relevant docs есть ниже top-5; возможна сильна...
6,exact,какая версия webOS или Tizen нужна для приложе...,[146],"[66, 86, 40, 172, 129]","[66, 86, 40, 172, 129, 146, 139, 72, 47, 30, 1...",0.0,1.0,{146: 6},relevant docs есть ниже top-5; dense-поиск мож...
7,multi,"на новом телевизоре пишет, что достигнут лимит...","[159, 166]","[66, 176, 131, 129, 72]","[66, 176, 131, 129, 72, 41, 173, 166, 47, 33, ...",0.0,1.0,"{159: 14, 166: 8}",relevant docs есть ниже top-5; multi-запрос мо...
8,multi,за границей при запуске фильма появляется P-10...,"[20, 47]","[90, 51, 110, 49, 138]","[90, 51, 110, 49, 138, 47, 129, 137, 20, 160, ...",0.0,1.0,"{20: 9, 47: 6}",relevant docs есть ниже top-5; multi-запрос мо...
9,multi,хочу смотреть в 4K — какой нужен тариф и какая...,"[101, 113]","[17, 45, 110, 24, 152]","[17, 45, 110, 24, 152, 168, 113, 101, 56, 144,...",0.0,1.0,"{101: 8, 113: 7}",relevant docs есть ниже top-5; multi-запрос мо...


### Вывод по шагу 4

Шаг 4 добавляет итоговый dense baseline для выбранной стратегии чанкинга, диагностический порог по распределению similarity score, выдачу top-K чанков с метаданными, сравнение обычного chunk-level top-20 с doc-level overfetch top-50/top-100, пример фильтрации по категории `Подписка и оплата`, метрики по типам запросов и таблицу проблемных запросов. Если `recall@20` после overfetch остаётся ниже целевого уровня, следующий шаг — улучшать candidate generation через BM25, RRF и query transformations, а reranker подключать только после попадания релевантных документов в candidate pool.

## Шаг 5. Улучшение retrieval и выбор итоговой схемы поиска

На этом шаге улучшаем candidate generation относительно dense-only baseline: добавляем структурированный `searchable_text`, BM25, dense+BM25 RRF и простые rule-based трансформации запросов.


In [20]:
from rank_bm25 import BM25Okapi

STEP5_CHUNK_K = 100
STEP5_DOC_K = 20
RRF_K = 60

def build_searchable_text(title: str, category: str, text: str) -> str:
    """Builds structured text used only for retrieval indexing.

    Args:
        title: Document title copied from metadata.
        category: Document category copied from metadata.
        text: Original chunk text preserved as `page_content`.

    Returns:
        Structured searchable text with title, category and chunk content.
    """
    parts = [
        f"\u0417\u0430\u0433\u043e\u043b\u043e\u0432\u043e\u043a: {title}",
        f"\u041a\u0430\u0442\u0435\u0433\u043e\u0440\u0438\u044f: {category}",
        f"\u0422\u0435\u043a\u0441\u0442: {text}",
    ]
    return "\n".join(parts)


def add_searchable_text(chunks):
    """Adds `searchable_text` while preserving original `page_content`.

    Args:
        chunks: Chunk dictionaries with title, category and page_content.

    Returns:
        New list of chunk dictionaries enriched with `searchable_text`.
    """
    enhanced = []
    for chunk in chunks:
        item = dict(chunk)
        item["searchable_text"] = build_searchable_text(
            title=item["title"],
            category=item["category"],
            text=item["page_content"],
        )
        enhanced.append(item)
    return enhanced


enhanced_chunks = add_searchable_text(selected_chunks)
enhanced_embeddings = encode_passages([c["searchable_text"] for c in enhanced_chunks])
enhanced_index = build_faiss_index(enhanced_embeddings)

print("Enhanced chunks:", len(enhanced_chunks))
print("Enhanced embeddings:", enhanced_embeddings.shape)
print("Enhanced index size:", enhanced_index.ntotal)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Enhanced chunks: 502
Enhanced embeddings: (502, 768)
Enhanced index size: 502


In [21]:
def dense_enhanced_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Searches the enhanced dense index by structured searchable text.

    Args:
        query_text: User query or transformed query text.
        chunk_k: Number of chunk candidates requested from FAISS.

    Returns:
        List of chunk hits with rank, score and original chunk metadata.
    """
    q_vec = encode_queries([query_text])[0]
    k = min(chunk_k, len(enhanced_chunks))
    scores, idx = enhanced_index.search(q_vec.reshape(1, -1), k)
    hits = []
    for rank, (score, chunk_idx) in enumerate(zip(scores[0], idx[0]), start=1):
        if chunk_idx < 0:
            continue
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": float(score),
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def doc_hits_from_chunk_hits(chunk_hits, doc_k: int = STEP5_DOC_K):
    """Aggregates chunk hits into ranked document hits.

    Args:
        chunk_hits: Chunk-level search results.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits with original text for downstream context building.
    """
    return aggregate_chunks_to_docs(chunk_hits, doc_k=doc_k)


def dense_enhanced_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Returns document hits from enhanced dense retrieval.

    Args:
        query_text: User query or transformed query text.
        chunk_k: Number of chunk candidates requested from FAISS.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits aggregated from enhanced dense chunk hits.
    """
    return doc_hits_from_chunk_hits(dense_enhanced_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

In [22]:
def tokenize_for_bm25(text: str):
    """Tokenizes Russian text for BM25 with simple lowercase regex rules.

    Args:
        text: Input document or query text.

    Returns:
        List of lowercase alphanumeric tokens with one-character tokens removed.
    """
    return [tok for tok in re.findall(r"[a-z\u0430-\u044f\u04510-9]+", text.lower()) if len(tok) > 1]


bm25_corpus = [tokenize_for_bm25(c["searchable_text"]) for c in enhanced_chunks]
bm25_index = BM25Okapi(bm25_corpus)

def bm25_chunk_search(query_text: str, chunk_k: int = STEP5_CHUNK_K):
    """Searches chunks with BM25 over enhanced searchable text.

    Args:
        query_text: User query or transformed query text.
        chunk_k: Number of chunk candidates to return.

    Returns:
        List of BM25 chunk hits with metadata and original page content.
    """
    tokens = tokenize_for_bm25(query_text)
    scores = bm25_index.get_scores(tokens)
    if len(scores) == 0:
        return []
    top_idx = np.argsort(scores)[::-1][:min(chunk_k, len(scores))]
    hits = []
    for rank, chunk_idx in enumerate(top_idx, start=1):
        score = float(scores[int(chunk_idx)])
        chunk = enhanced_chunks[int(chunk_idx)]
        hits.append({
            "rank": rank,
            "chunk_index": int(chunk_idx),
            "score": score,
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "category": chunk["category"],
            "chunk_id": chunk["chunk_id"],
            "page_content": chunk["page_content"],
        })
    return hits


def bm25_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Returns document hits from BM25 retrieval.

    Args:
        query_text: User query or transformed query text.
        chunk_k: Number of chunk candidates requested from BM25.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits aggregated from BM25 chunk hits.
    """
    return doc_hits_from_chunk_hits(bm25_chunk_search(query_text, chunk_k=chunk_k), doc_k=doc_k)

print("BM25 corpus size:", len(bm25_corpus))
print("Median BM25 tokens per chunk:", int(np.median([len(x) for x in bm25_corpus])))

BM25 corpus size: 502
Median BM25 tokens per chunk: 50


In [23]:
def reciprocal_rank_fusion(ranked_lists, k: int = RRF_K, doc_k: int = STEP5_DOC_K):
    """Combines several ranked document lists with Reciprocal Rank Fusion.

    Args:
        ranked_lists: Sequence of ranked document-hit lists.
        k: RRF smoothing constant.
        doc_k: Number of fused documents to return.

    Returns:
        Ranked fused document hits with `rrf_score` and original metadata.
    """
    fused = {}
    for ranked in ranked_lists:
        for rank, hit in enumerate(ranked, start=1):
            doc_id = hit["doc_id"]
            if doc_id not in fused:
                fused[doc_id] = dict(hit)
                fused[doc_id]["rrf_score"] = 0.0
                fused[doc_id]["source_count"] = 0
            fused[doc_id]["rrf_score"] += 1.0 / (k + rank)
            fused[doc_id]["source_count"] += 1
    rows = sorted(
        fused.values(),
        key=lambda row: (-row["rrf_score"], -row.get("source_count", 0), row.get("best_chunk_rank", row.get("rank", 9999))),
    )[:doc_k]
    for rank, row in enumerate(rows, start=1):
        row["doc_rank"] = rank
    return rows


def hybrid_rrf_doc_hits(query_text: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Returns document hits fused from enhanced dense and BM25 retrieval.

    Args:
        query_text: User query or transformed query text.
        chunk_k: Number of chunk candidates for each retriever.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits after dense+BM25 RRF fusion.
    """
    dense_docs = dense_enhanced_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    bm25_docs = bm25_doc_hits(query_text, chunk_k=chunk_k, doc_k=doc_k)
    return reciprocal_rank_fusion([dense_docs, bm25_docs], doc_k=doc_k)

In [24]:
QUERY_EXPANSION_RULES = {
    "\u043e\u043f\u043b\u0430\u0442": "\u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u043b\u0430\u0442\u0435\u0436 \u043a\u0430\u0440\u0442\u0430 \u0441\u043f\u0438\u0441\u0430\u043d\u0438\u0435 \u0447\u0435\u043a \u043a\u0432\u0438\u0442\u0430\u043d\u0446\u0438\u044f \u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430",
    "\u043a\u0430\u0440\u0442": "\u0431\u0430\u043d\u043a\u043e\u0432\u0441\u043a\u0430\u044f \u043a\u0430\u0440\u0442\u0430 \u043e\u043f\u043b\u0430\u0442\u0430 \u043f\u0440\u0438\u0432\u044f\u0437\u0430\u0442\u044c \u043e\u0442\u0432\u044f\u0437\u0430\u0442\u044c \u0441\u043f\u043e\u0441\u043e\u0431 \u043e\u043f\u043b\u0430\u0442\u044b",
    "\u043f\u043e\u0434\u043f\u0438\u0441": "\u043f\u043e\u0434\u043f\u0438\u0441\u043a\u0430 \u0442\u0430\u0440\u0438\u0444 \u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u0430\u0432\u0442\u043e\u043f\u0440\u043e\u0434\u043b\u0435\u043d\u0438\u0435 \u043e\u0442\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043e\u0448\u0438\u0431": "\u043e\u0448\u0438\u0431\u043a\u0430 \u043a\u043e\u0434 \u0432\u043e\u0441\u043f\u0440\u043e\u0438\u0437\u0432\u0435\u0434\u0435\u043d\u0438\u0435 \u043f\u043b\u0435\u0435\u0440 \u043f\u0440\u043e\u0431\u043b\u0435\u043c\u0430",
    "\u0443\u0441\u0442\u0440\u043e\u0439": "\u0443\u0441\u0442\u0440\u043e\u0439\u0441\u0442\u0432\u043e \u0442\u0435\u043b\u0435\u0432\u0438\u0437\u043e\u0440 \u0441\u043c\u0430\u0440\u0442\u0444\u043e\u043d \u043f\u043b\u0430\u043d\u0448\u0435\u0442 \u043f\u0440\u0438\u0441\u0442\u0430\u0432\u043a\u0430 \u043f\u043e\u0434\u043a\u043b\u044e\u0447\u0438\u0442\u044c",
    "\u043f\u0440\u043e\u0444\u0438\u043b": "\u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0434\u0435\u0442\u0441\u043a\u0438\u0439 \u043f\u0440\u043e\u0444\u0438\u043b\u044c \u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u043f\u043e\u043b\u044c\u0437\u043e\u0432\u0430\u0442\u0435\u043b\u044c",
    "\u0430\u043a\u043a\u0430\u0443\u043d\u0442": "\u0430\u043a\u043a\u0430\u0443\u043d\u0442 \u0432\u0445\u043e\u0434 \u043f\u0430\u0440\u043e\u043b\u044c \u043f\u043e\u0447\u0442\u0430 \u0442\u0435\u043b\u0435\u0444\u043e\u043d \u043f\u0440\u043e\u0444\u0438\u043b\u044c",
}

def expand_query_rule_based(query_text: str):
    """Expands a semantic query with compact domain synonyms.

    Args:
        query_text: Original user query.

    Returns:
        Expanded query string. If no rule matches, returns the original query.
    """
    lowered = query_text.lower()
    additions = []
    for marker, expansion in QUERY_EXPANSION_RULES.items():
        if marker in lowered:
            additions.append(expansion)
    if not additions:
        return query_text
    return query_text + " " + " ".join(dict.fromkeys(" ".join(additions).split()))


def decompose_query_rule_based(query_text: str):
    """Splits multi-intent queries into lightweight subqueries.

    Args:
        query_text: Original user query.

    Returns:
        List containing the original query and rule-based subqueries.
    """
    split_pattern = "\\s+(?:\u0438\u043b\u0438|\u0430\\s+\u0442\u0430\u043a\u0436\u0435|\u043f\u043b\u044e\u0441|\u0438)\\s+|[,;]"
    parts = [p.strip(" ,.;:!?()") for p in re.split(split_pattern, query_text, flags=re.IGNORECASE)]
    parts = [p for p in parts if len(p) >= 8]
    queries_out = [query_text]
    for part in parts:
        if part.lower() != query_text.lower() and part not in queries_out:
            queries_out.append(part)
    return queries_out


def hybrid_rrf_expanded_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Runs hybrid retrieval with expansion for semantic queries.

    Args:
        query_text: Original user query.
        kind: Query kind from the evaluation set.
        chunk_k: Number of chunk candidates for each retriever.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits after optional expansion and RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


def hybrid_rrf_expanded_decomposed_doc_hits(query_text: str, kind: str, chunk_k: int = STEP5_CHUNK_K, doc_k: int = STEP5_DOC_K):
    """Runs hybrid retrieval with semantic expansion and multi-query decomposition.

    Args:
        query_text: Original user query.
        kind: Query kind from the evaluation set.
        chunk_k: Number of chunk candidates for each retriever.
        doc_k: Number of final unique documents.

    Returns:
        Ranked document hits after optional transformations and RRF.
    """
    variants = [query_text]
    if kind == "semantic":
        expanded = expand_query_rule_based(query_text)
        if expanded != query_text:
            variants.append(expanded)
    if kind == "multi":
        variants.extend(decompose_query_rule_based(query_text)[1:])
    variants = list(dict.fromkeys(variants))
    ranked_lists = [hybrid_rrf_doc_hits(v, chunk_k=chunk_k, doc_k=doc_k) for v in variants]
    return reciprocal_rank_fusion(ranked_lists, doc_k=doc_k)


In [25]:
def evaluate_retriever_variant(variant: str, retrieve_docs_fn, doc_k: int = STEP5_DOC_K):
    """Evaluates a document-level retriever on non-gap evaluation queries.

    Args:
        variant: Human-readable variant name.
        retrieve_docs_fn: Function accepting one query dict and returning document hits.
        doc_k: Number of top documents used for metrics.

    Returns:
        `pandas.DataFrame` with per-query retrieval metrics.
    """
    rows = []
    for q in eval_queries:
        doc_hits = retrieve_docs_fn(q)[:doc_k]
        found_docs = [row["doc_id"] for row in doc_hits]
        row = {
            "variant": variant,
            "kind": q["kind"],
            "query": q["query"],
            "relevant_docs": q["relevant_docs"],
            "found_doc_ids": found_docs,
            "precision@5": precision_at_k(found_docs[:5], q["relevant_docs"]),
            "recall@5": recall_at_k(found_docs[:5], q["relevant_docs"]),
            "MRR": mrr(found_docs, q["relevant_docs"]),
        }
        for k in RECALL_KS:
            row[f"recall@{k}"] = recall_at_k(found_docs[:k], q["relevant_docs"])
        rows.append(row)
    return pd.DataFrame(rows)


step5_results = [
    evaluate_retriever_variant(
        "dense_enhanced_chunk100_doc20",
        lambda q: dense_enhanced_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "bm25_chunk100_doc20",
        lambda q: bm25_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_dense_bm25_rrf_doc20",
        lambda q: hybrid_rrf_doc_hits(q["query"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_semantic_doc20",
        lambda q: hybrid_rrf_expanded_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
    evaluate_retriever_variant(
        "hybrid_rrf_expanded_decomposed_doc20",
        lambda q: hybrid_rrf_expanded_decomposed_doc_hits(q["query"], q["kind"], chunk_k=STEP5_CHUNK_K, doc_k=STEP5_DOC_K),
    ),
]

step5_diagnostics = pd.concat(step5_results, ignore_index=True)
step5_summary = step5_diagnostics.groupby("variant")[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index().sort_values(["recall@5", "MRR", "recall@20"], ascending=False)
step5_summary

,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
1,dense_enhanced_chunk100_doc20,0.175000,0.687500,0.578407,0.339286,0.616071,0.794643,0.928571
4,hybrid_rrf_expanded_semantic_doc20,0.157143,0.616071,0.550128,0.330357,0.589286,0.758929,0.839286
2,hybrid_dense_bm25_rrf_doc20,0.157143,0.616071,0.548370,0.330357,0.589286,0.758929,0.857143
3,hybrid_rrf_expanded_decomposed_doc20,0.150000,0.598214,0.534239,0.330357,0.553571,0.758929,0.839286
0,bm25_chunk100_doc20,0.125000,0.491071,0.419227,0.232143,0.410714,0.589286,0.669643


In [26]:
step5_by_kind = step5_diagnostics.groupby(["variant", "kind"])[[
    "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]].mean().reset_index()
step5_by_kind.sort_values(["variant", "kind"])

,variant,kind,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,bm25_chunk100_doc20,exact,0.186667,0.933333,0.726190,0.533333,0.866667,0.966667,0.966667
1,bm25_chunk100_doc20,multi,0.187500,0.468750,0.550000,0.187500,0.375000,0.593750,0.718750
2,bm25_chunk100_doc20,semantic,0.048000,0.240000,0.151355,0.080000,0.160000,0.360000,0.460000
3,dense_enhanced_chunk100_doc20,exact,0.200000,0.966667,0.757778,0.600000,0.900000,0.966667,1.000000
4,dense_enhanced_chunk100_doc20,multi,0.237500,0.593750,0.657532,0.250000,0.500000,0.625000,0.843750
5,dense_enhanced_chunk100_doc20,semantic,0.120000,0.580000,0.420144,0.240000,0.520000,0.800000,0.940000
6,hybrid_dense_bm25_rrf_doc20,exact,0.200000,0.966667,0.800000,0.666667,0.966667,0.966667,0.966667
7,hybrid_dense_bm25_rrf_doc20,multi,0.212500,0.531250,0.625248,0.218750,0.500000,0.625000,0.750000
8,hybrid_dense_bm25_rrf_doc20,semantic,0.096000,0.460000,0.348190,0.200000,0.420000,0.720000,0.860000
9,hybrid_rrf_expanded_decomposed_doc20,exact,0.200000,0.966667,0.800000,0.666667,0.966667,0.966667,0.966667


In [27]:
baseline_best_name = "dense_chunk100_doc20"
baseline_best_metrics = baseline_summary[baseline_summary["variant"] == baseline_best_name].copy()
if baseline_best_metrics.empty:
    baseline_best_metrics = baseline_summary.head(1).copy()
    baseline_best_name = baseline_best_metrics.iloc[0]["variant"]

best_step5_variant = step5_summary.iloc[0]["variant"]
best_step5_metrics = step5_summary[step5_summary["variant"] == best_step5_variant].copy()

comparison_before_after = pd.concat([
    baseline_best_metrics.assign(stage="before_step5"),
    best_step5_metrics.assign(stage="after_step5"),
], ignore_index=True)[[
    "stage", "variant", "precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"
]]

metric_cols = ["precision@5", "recall@5", "MRR", "recall@1", "recall@3", "recall@10", "recall@20"]
improvements = {
    f"delta_{col}": float(best_step5_metrics.iloc[0][col] - baseline_best_metrics.iloc[0][col])
    for col in metric_cols
}

print("Baseline variant:", baseline_best_name)
print("Best step 5 variant:", best_step5_variant)
print("Metric deltas:", {k: round(v, 6) for k, v in improvements.items()})
comparison_before_after

Baseline variant: dense_chunk100_doc20
Best step 5 variant: dense_enhanced_chunk100_doc20
Metric deltas: {'delta_precision@5': 0.021429, 'delta_recall@5': 0.098214, 'delta_MRR': 0.111181, 'delta_recall@1': 0.080357, 'delta_recall@3': 0.116071, 'delta_recall@10': 0.107143, 'delta_recall@20': 0.098214}


,stage,variant,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,before_step5,dense_chunk100_doc20,0.153571,0.589286,0.467225,0.258929,0.500000,0.687500,0.830357
1,after_step5,dense_enhanced_chunk100_doc20,0.175000,0.687500,0.578407,0.339286,0.616071,0.794643,0.928571


In [28]:
best_step5_rows = step5_diagnostics[step5_diagnostics["variant"] == best_step5_variant].copy()
remaining_problem_queries = best_step5_rows[best_step5_rows["recall@5"] < 1.0].sort_values([
    "recall@5", "recall@20", "kind"
]).reset_index(drop=True)

best_recall5 = float(best_step5_metrics.iloc[0]["recall@5"])
best_recall20 = float(best_step5_metrics.iloc[0]["recall@20"])
if best_recall20 >= 0.90 and best_recall5 < 0.85:
    next_retrieval_step = "connect reranker: candidate pool is broad enough, but top-5 ranking is weak"
elif best_recall20 < 0.90:
    next_retrieval_step = "continue candidate generation: recall@20 is still below 0.90"
else:
    next_retrieval_step = "move to answer generation and LLM judge"

print("Problem queries for best step 5 variant with recall@5 < 1.0:", len(remaining_problem_queries))
print("Next retrieval decision:", next_retrieval_step)
remaining_problem_queries.head(15)

Problem queries for best step 5 variant with recall@5 < 1.0: 23
Next retrieval decision: connect reranker: candidate pool is broad enough, but top-5 ranking is weak


,variant,kind,query,relevant_docs,found_doc_ids,precision@5,recall@5,MRR,recall@1,recall@3,recall@10,recall@20
0,dense_enhanced_chunk100_doc20,semantic,"боюсь, что ребёнок случайно купит фильм с моей...",[140],"[108, 84, 73, 171, 159, 20, 116, 54, 173, 124,...",0.0,0.0,0.000000,0.0,0.0,0.0,0.0
1,dense_enhanced_chunk100_doc20,multi,заплатил за новинку отдельно — сколько она у м...,"[73, 140]","[116, 106, 111, 141, 3, 71, 159, 29, 24, 46, 8...",0.0,0.0,0.076923,0.0,0.0,0.0,0.5
2,dense_enhanced_chunk100_doc20,multi,хочу смотреть в 4K — какой нужен тариф и какая...,"[101, 113]","[17, 45, 24, 110, 152, 88, 56, 144, 159, 60, 1...",0.0,0.0,0.076923,0.0,0.0,0.0,1.0
3,dense_enhanced_chunk100_doc20,semantic,"включаю кино — на экране всё идёт, а из колоно...",[112],"[159, 150, 132, 50, 69, 99, 112, 22, 101, 1, 1...",0.0,0.0,0.142857,0.0,0.0,1.0,1.0
4,dense_enhanced_chunk100_doc20,semantic,"как сделать, чтобы деньги сами не снимались ка...","[61, 64]","[114, 164, 5, 180, 162, 99, 155, 64, 116, 108,...",0.0,0.0,0.125000,0.0,0.0,0.5,1.0
5,dense_enhanced_chunk100_doc20,semantic,скачанная серия перестала открываться без инте...,[49],"[144, 109, 47, 94, 99, 49, 67, 90, 20, 51, 177...",0.0,0.0,0.166667,0.0,0.0,1.0,1.0
6,dense_enhanced_chunk100_doc20,semantic,"хочу убрать из ленты фильмы, которые уже смотрел",[97],"[20, 81, 99, 119, 73, 159, 157, 70, 97, 108, 1...",0.0,0.0,0.111111,0.0,0.0,1.0,1.0
7,dense_enhanced_chunk100_doc20,semantic,"не помню название, помню только актёра",[158],"[38, 111, 55, 124, 112, 71, 159, 43, 99, 46, 7...",0.0,0.0,0.076923,0.0,0.0,0.0,1.0
8,dense_enhanced_chunk100_doc20,semantic,"видят ли другие люди, что я поставил фильму лайк",[135],"[99, 157, 134, 173, 27, 159, 20, 32, 111, 38, ...",0.0,0.0,0.058824,0.0,0.0,0.0,1.0
9,dense_enhanced_chunk100_doc20,semantic,"в приложении всё включено, а на телефон ничего...",[124],"[72, 66, 94, 172, 146, 41, 54, 30, 117, 131, 1...",0.0,0.0,0.055556,0.0,0.0,0.0,1.0


### Вывод по шагу 5

Шаг 5 сравнивает улучшенные варианты retrieval на том же evaluation-наборе без `gap`-запросов. Итоговая схема выбирается по `recall@5`, затем `MRR` и `recall@20`.
